In [1]:
import os
import pandas as pd

# Paths
base_dir = r'../dataset/Dataset'
image_dir = os.path.join(base_dir, 'Memes')  # Path to the extracted "Memes" folder
input_excel = os.path.join(base_dir, 'multi-sent.xlsx')  # Path to the Excel file
output_csv = "../working/image_caption_labels.csv"  # Path for the output CSV file

# Step 1: Get all image files from the Memes folder   
image_files = os.listdir(image_dir)

# Step 2: Load the Excel file
df = pd.read_excel(input_excel)

# Step 3: Create a new list to store data for CSV
image_data = []

# Step 4: Iterate through the Excel data and match image_name with files
for index, row in df.iterrows():
    image_name = row['image_name']  # Assuming the column name is 'image_name'

    # Check if the image exists in the folder
    if image_name in image_files:
        # Construct the full image path
        image_path = os.path.join(image_dir, image_name)

        # Extract Captions and Label_Sentiment (adjust column names if necessary)
        caption = row['Captions']  # Assuming 'Captions' is the column name for captions
        label_sentiment = row['Label_Sentiment']  # Assuming 'Label_Sentiment' is the column name

        # Append the row with image path, image name, captions, and sentiment label
        image_data.append([image_path, image_name, caption, label_sentiment])

# Step 5: Create a DataFrame for the matched data
image_df = pd.DataFrame(image_data, columns=['Image_path', 'image_name', 'Captions', 'Label_Sentiment'])

# Step 6: Save the DataFrame to a CSV file
image_df.to_csv(output_csv, index=False)

print(f"CSV file saved at {output_csv}")


CSV file saved at ../working/image_caption_labels.csv


In [2]:
import pandas as pd

# Path to the CSV file (replace this with the actual path if different)
csv_file = "../working/image_caption_labels.csv"

# Step 1: Load the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Step 2: Print the DataFrame to show the contents
df.head()


,Image_path,image_name,Captions,Label_Sentiment
0,../dataset/Dataset\Memes\tangaila (1).jpg,tangaila (1).jpg,"আম্মাঃ HSC চলে আসছে , এখন থেকে তোর মোবাইল , ল্...",positive
1,../dataset/Dataset\Memes\tangaila (2).jpg,tangaila (2).jpg,WHEN YOUR COUSINS TAKES YOU TO THE DHAN KHET A...,negative
2,../dataset/Dataset\Memes\tangaila (3).jpg,tangaila (3).jpg,WHEN HE SAID 10 MINUTES BUT IT WAS ONLY 2 MINUTES,negative
3,../dataset/Dataset\Memes\tangaila (4).jpg,tangaila (4).jpg,SHE - I CAN'T BE WITH YOU -তবে শেষবারের মত দ...,negative
4,../dataset/Dataset\Memes\tangaila (5).jpg,tangaila (5).jpg,"যখন কোন Teacher বলে ""সত্যটা বলো, তাহলে আর কি...",negative


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 2: Split the data into train (70%), test (20%), and validation (10%) using stratified splitting
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['Label_Sentiment'], random_state=42)
test_df, val_df = train_test_split(temp_df, test_size=1/3, stratify=temp_df['Label_Sentiment'], random_state=42)

# Step 3: Save the split data into separate CSV files
train_df.to_csv('../working/train.csv', index=False)
test_df.to_csv('../working/test.csv', index=False)
val_df.to_csv('../working/validation.csv', index=False)

# Print the shapes of the resulting datasets for verification
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Validation shape: {val_df.shape}")


Train shape: (3057, 4)
Test shape: (874, 4)
Validation shape: (437, 4)


In [4]:
import pandas as pd
import re
import string

# Function to remove punctuation (preserve Bangla characters)
def remove_punctuation(text):
    if pd.isna(text):
        return text
    return text.translate(str.maketrans('', '', string.punctuation))

# Function to remove extra whitespace
def remove_whitespace(text):
    if pd.isna(text):
        return text
    return " ".join(text.split())

# Function to remove emojis
def remove_emojis(text):
    if pd.isna(text):
        return text
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Function to remove URLs
def remove_urls(text):
    if pd.isna(text):
        return text
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

# Function to remove HTML tags
def remove_html(text):
    if pd.isna(text):
        return text
    html_pattern = re.compile(r'<.*?>')
    return html_pattern.sub(r'', text)

# Function to remove special characters (preserve Bangla characters)
def remove_special_characters(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^A-Za-z0-9\s\u0980-\u09FF]', '', text)

# Combine all cleaning functions
def clean_text(text):
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_emojis(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = remove_whitespace(text)
    return text

# Mapping categories to integers
category_mapping = {
    'positive': 0,
    'negative': 1,
    'neutral': 2,
}

# Paths to the previously saved CSVs
csv_paths = {
    'Train': '../working/train.csv',
    'Test': '../working/test.csv',
    'Validation': '../working/validation.csv'
}

# Output paths for the cleaned CSVs
cleaned_output_paths = {
    'Train': '../working/train_cleaned.csv',
    'Test': '../working/test_cleaned.csv',
    'Validation': '../working/validation_cleaned.csv'
}

# Text columns to clean
text_columns = ['Captions', 'Label_Sentiment']

# Loop through each dataset
for key in csv_paths:
    # Load the dataset
    df = pd.read_csv(csv_paths[key])

    # Apply cleaning to all relevant text columns
    for column in text_columns:
        df[column] = df[column].astype(str).apply(clean_text)

    # Map the 'Label_Sentiment' column to integers
    df['Label_Sentiment'] = df['Label_Sentiment'].map(category_mapping)

    # Add a 'label' column (same as 'Label_Sentiment' for now)
    df['label'] = df['Label_Sentiment']

    # Display the cleaned dataframe
    print(f"Cleaned {key} dataframe:")
    print(df.head())

    # Save the cleaned dataframe to a new CSV file
    df.to_csv(cleaned_output_paths[key], index=False)
    print(f"Cleaned dataframe saved to {cleaned_output_paths[key]}\n")


Cleaned Train dataframe:
                                        Image_path              image_name  \
0  ../dataset/Dataset\Memes\bamboo-vaiya (115).jpg  bamboo-vaiya (115).jpg   
1            ../dataset/Dataset\Memes\KAM (61).png            KAM (61).png   
2           ../dataset/Dataset\Memes\KAM (345).jpg           KAM (345).jpg   
3           ../dataset/Dataset\Memes\KAM (572).jpg           KAM (572).jpg   
4           ../dataset/Dataset\Memes\KAM (714).jpg           KAM (714).jpg   

                                            Captions  Label_Sentiment  label  
0  WHEN GIRL CHANGES HER RELATIONSHIP STATUS TO S...                1      1  
1  ভাল ফ্যাকাল্টি পাইলে আমিও ভাল গ্রেড উঠাইতাম কি...                0      0  
2  BREAK UP এরপর মেয়েরা যে গান গায় এনালগ যুগঃ যেও...                1      1  
3  কি হয়সে ভাই আপনার মন খারাপ কেন সেই কখন থেকে মো...                0      0  
4  পড়তে বসার সময় বসার 10 মিনিট পর বসার 20 মিনিট...                0      0  
Cleaned dataframe saved to ../wo